# Polygon plot

In [8]:
### import necessary libraries
import pandas as pd
import geopandas as gpd
import geojson
import os

In [ ]:
name_dir = ''

from module.config_local import sample_name_import, dir_processed,dir_raw

samples_ids = sample_name_import(name_dir)

## Geojson preprocessing

### Import cells bondaries and create Geojson file (not resegmented)

In [9]:
for sample_id in samples_ids:
    ### Get cell and nucleus boundaries
    cells = pd.read_parquet(f"{dir_raw}/{sample_id}/cell_boundaries.parquet")
    
    cells.groupby("cell_id")[['vertex_x', 'vertex_y']].agg({"vertex_x": list, "vertex_y": list}).reset_index().rename(columns={"vertex_x": "xs", "vertex_y": "ys"})
    cells['cell_id'] = sample_id + '_' + cells['cell_id']
    
    # Group the dataframe by the "Selection" column
    grouped = cells.groupby('cell_id')
    features = []
    
    for cell_id, group in grouped:
        coordinates = [(x, y) for x, y in zip(group['vertex_x'], group['vertex_y'])]
        if coordinates[0] != coordinates[-1]:
            coordinates.append(coordinates[0])
        
        polygon = geojson.Polygon([coordinates])
        feature = geojson.Feature(geometry=polygon, properties={"cell": cell_id})
        features.append(feature)
    
    feature_collection = geojson.FeatureCollection(features)
    
    try:
        with open(f'{dir_processed}/coordinates/polygons/{sample_id}_cells.geojson', 'w') as f:
            geojson.dump(feature_collection, f)
    except:
        os.mkdirs(f'{dir_processed}/coordinates/polygons/')
        with open(f'{dir_processed}/coordinates/polygons/{sample_id}_cells.geojson', 'w') as f:
            geojson.dump(feature_collection, f)        
    
    print(f"{sample_id}: GeoJSON saved")

hLiver-cancer: GeoJSON saved
hLiver-nondiseased: GeoJSON saved


### Update Geojson file (resegmented)

In [ ]:
### sample
name_dir = ''

from module.config_local import sample_name_import, dir_processed,dir_raw

samples_ids = sample_name_import(name_dir)

path_to_resegment = ''

In [ ]:
for sample_id in sample_ids:
    cells_geo = gpd.read_file(f'{path_to_resegment}/{sample_id}.geojson') ### if resegment
    cells_geo['cell'] = sample_id + '_' + cells_geo['id'] ### Rsegment only
    with open(f'{dir_processed}/coordinates/polygons/{sample_id}_cells.geojson', 'w') as f:
        geojson.dump(cells_geo, f)